In [ ]:
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression

In [ ]:
# upload the usgs and foothills influent data'
usgs = pd.read_csv(r"C:\Users\jslawson\OneDrive - Denver Water\SCO\Source_water_early_warning_systems\Data\USGS_South_Platte.csv")
fth_inf = pd.read_csv(r"C:\Users\jslawson\OneDrive - Denver Water\SCO\DBP_Python2026\Clean_CSV\PL-FTH-INF_cleaned.csv")
fth_hw = pd.read_csv(r"C:\Users\jslawson\OneDrive - Denver Water\SCO\DBP_Python2026\Clean_CSV\PL-FTH-HW_cleaned.csv")
# Now for the DWR flow data
dwr = pd.read_csv(r"C:\Users\jslawson\OneDrive - Denver Water\SCO\Source_water_early_warning_systems\Data\SouthPlatteTelemetry.csv")
# NOAA meteo data at strontia
precip = pd.read_csv(r"C:\Users\jslawson\OneDrive - Denver Water\SCO\Source_water_early_warning_systems\Data\USC00058022.csv")

In [ ]:
# Foothills data formatting - pivot and keep only Alk
fth_inf['DATE'] = pd.to_datetime(fth_inf['SAMPLED_DATE'])
fth_inf.set_index('DATE', inplace=True)
fth_inf['FORMATTED_ENTRY'] = pd.to_numeric(fth_inf['FORMATTED_ENTRY'], errors="coerce")

# Pivot
fth_inf_pivot = fth_inf.pivot_table(
    index='DATE',
    columns='ANALYTE',
    values='FORMATTED_ENTRY',
    aggfunc='median' 
)

# Trim to April 2022
fth_inf_pivot = fth_inf_pivot[(fth_inf_pivot.index.month >= 4) & (fth_inf_pivot.index.year >= 2022)].copy()

# Drop the mult-index
fth_inf_pivot.columns.name = None

In [ ]:
# Keep only Alk
cols_to_keep = ['Alk']
fth_inf_pivot = fth_inf_pivot[cols_to_keep].copy()

In [ ]:
# Same thing for HW, then concatenate into a single site/df
# Foothills data formatting - pivot and keep only Alk
fth_hw['DATE'] = pd.to_datetime(fth_hw['SAMPLED_DATE'])
fth_hw.set_index('DATE', inplace=True)
fth_hw['FORMATTED_ENTRY'] = pd.to_numeric(fth_hw['FORMATTED_ENTRY'], errors="coerce")

# Pivot
fth_hw_pivot = fth_hw.pivot_table(
    index='DATE',
    columns='ANALYTE',
    values='FORMATTED_ENTRY',
    aggfunc='median' 
)

# Trim to April 2022
fth_hw_pivot = fth_hw_pivot[(fth_hw_pivot.index.month >= 4) & (fth_hw_pivot.index.year >= 2022)].copy()

# Drop the mult-index
fth_hw_pivot.columns.name = None

fth_hw_pivot = fth_hw_pivot[cols_to_keep].copy()

In [ ]:
# Concatenate foothills data
combined_fth = pd.concat([fth_inf_pivot, fth_hw_pivot])
combined_fth = combined_fth.groupby(level=0).median().dropna()
combined_fth

In [ ]:
# Set USGS index to datetime index, remove timezone
usgs['DATE'] = pd.to_datetime(usgs['Date']).dt.tz_localize(None)
usgs = usgs.drop(columns = 'Date')
usgs.set_index('DATE', inplace=True)

# Lag the USGS dataframe by 4 days
usgs_lagged = usgs.shift(4, freq='D')

# join dfs based on shared index
combined_df = combined_fth.join(usgs_lagged, how="left")

In [ ]:
# Format DWR flow data, lag by 4 days, and merge
dwr['DATE'] = pd.to_datetime(dwr['Date'])
dwr = dwr.drop(columns = 'Date')
dwr.set_index('DATE', inplace=True)
dwr_lagged = dwr.shift(4, freq='D')
combined_df = combined_df.join(dwr_lagged, how="left")

In [ ]:
# Format NOAA precip data, lag by 6 days, and merge
precip['DATE'] = pd.to_datetime(precip['DATE'])
precip.set_index('DATE', inplace=True)
precip_lagged = precip.shift(6, freq='D')
combined_df = combined_df.join(precip_lagged, how="left")

In [ ]:
# Add a month column to use as a predictor
combined_df['Month'] = combined_df.index.month

# Create the angle in radians (2 * pi * month / 12)
# We subtract 1 so January starts at 0
month_radians = 2 * np.pi * (combined_df['Month'] - 1) / 12

# Generate Sine and Cosine components
combined_df['month_sin'] = np.sin(month_radians)
combined_df['month_cos'] = np.cos(month_radians)

# Drop rows where Alk is null
combined_df_alk = combined_df.dropna(subset=['Alk']).copy()

In [ ]:
# Adding more engineered features for prediction 
# Is the river rising or falling?
combined_df_alk['flow_delta'] = combined_df_alk['Flow_CFS'].diff() 

# 7-day rolling average (the 'Baseline' state of the watershed)
combined_df_alk['flow_7day_avg'] = combined_df_alk['Flow_CFS'].rolling(window=7).mean()

# Create a turbidity 'Trend' feature
combined_df_alk['turb_3day'] = combined_df_alk['Turbidity_Median'].rolling(window=3).mean()

# Create a doy feature
day_of_year = combined_df.index.dayofyear
combined_df_alk['doy_sin'] = np.sin(2 * np.pi * day_of_year / 365)
combined_df_alk['doy_cos'] = np.cos(2 * np.pi * day_of_year / 365)

# turbidity "loading" feature
combined_df_alk['turb_flow'] = combined_df_alk['turb_3day'] * combined_df_alk['flow_7day_avg']

# Experimental precip features
combined_df_alk['precip_7day'] = combined_df_alk['PRCP'].rolling(window=7).mean()
combined_df_alk['precip_3day'] = combined_df_alk['PRCP'].rolling(window=3).mean()

In [ ]:
combined_df_alk

In [ ]:
corr_matrix = combined_df_alk.corr(numeric_only=True, method='spearman')

# Keep only correlations with TOC
toc_corr = corr_matrix[['Alk']].sort_values('Alk', ascending=False)

plt.figure(figsize=(8, 10))
sns.heatmap(
    toc_corr,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    center=0
)

plt.title('Spearman Correlation with Alk')
plt.tight_layout()
plt.savefig(r"C:\Users\jslawson\OneDrive - Denver Water\SCO\Source_water_early_warning_systems\Data\Figures\AlkalinityCorrelationMatrix_spearman.png")
plt.show()

In [ ]:
# Building a quick linear regression with conductivity to test the final model
# 1. Define the model features
feature = ['Specific_Cond_Mean']
target = 'Alk' 

# Select only features + the target, then drop rows with NaNs
df_linear_model = combined_df_alk[feature + [target]].dropna()

# Set X and y
X = df_linear_model[feature]
y = df_linear_model[target]

# 2. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.45, shuffle = False
)

# 3. Initialize the model
model = LinearRegression()

# 4. Train the model using the training data
model.fit(X_train, y_train)

# 5. Make predictions on the test data
y_pred = model.predict(X_test)

# 6. Extract learned mathematical properties
print(f"Slope (Coefficient): {model.coef_[0]:.4f}")
print(f"Y-Intercept: {model.intercept_:.4f}")

# 7. Evaluate model performance
print(f"Root Mean Squared Error (RMSE): {root_mean_squared_error(y_test, y_pred):.4f}")
print(f"R-squared Score: {r2_score(y_test, y_pred):.4f}")

# 8. Plot the model
plt.figure(figsize=(12, 6))

# Plot actual values
plt.scatter(y_test.index, y_test.values, label='Measured Foothills Influent Alkalinity', 
         color='blue', marker='o', alpha=0.8)

# Plot predicted values
plt.plot(y_test.index, y_pred, label='Soft Sensor Predicted Alkalinity (Linear regression)', 
         color='orange', linestyle='--', linewidth=2)

plt.title(f"Alkalinity Soft Sensor Performance")
plt.xlabel("Date")
plt.ylabel("Alkalinity (mg/L)")
plt.legend()
plt.grid(True, alpha=0.3)

# Tighten layout to prevent clipping
plt.tight_layout()
plt.show()

In [ ]:
# Assign predictors and target columns
features = ['Specific_Cond_Mean', 'pH_Median', 'month_sin', 'month_cos', 'flow_7day_avg', 'turb_3day', 
            'turb_flow', 'Dissolved_Oxygen_Mean']
target = 'Alk' 

# Select only features + the target, then drop rows with NaNs
df_model = combined_df_alk[features + [target]].dropna()

# Set X and y
X = df_model[features]
y = df_model[target]

In [ ]:
df_model.tail()

In [ ]:
print("Starting Grid Search...")

# 55/45 Split
X_train_cv, X_test, y_train_cv, y_test = train_test_split(
    X, y, test_size=0.45, shuffle=False
)

# Create weights
weights = np.where(y_train_cv < 60.0, 1.5, 1.0)

# Setup the Grid Search
# We focus on depth and leaf size to pull that biased prediction line down
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'min_samples_leaf': [5, 10, 20],
    'max_features': [1.0, 'sqrt'] # 1.0 uses all features, sqrt uses a subset
}

tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=tscv,
    n_jobs=-1,
    verbose=1
)

# Fit Grid Search on the Training set
grid_search.fit(X_train_cv, y_train_cv, sample_weight = weights)

# Extracting mean, std dev, and last fold R2 for the best model
best_index = grid_search.best_index_
mean_cv_r2 = grid_search.cv_results_['mean_test_score'][best_index]
std_cv_r2 = grid_search.cv_results_['std_test_score'][best_index]

# Calculate the index of the last split
last_split_index = grid_search.n_splits_ - 1

# Extract the score for the last fold at the best_index
last_fold_r2 = grid_search.cv_results_[f'split{last_split_index}_test_score'][best_index]

# Print the results
print(f"\nCross-Validation Results")
print(f"Best Mean R2:  {mean_cv_r2:.4f}")
print(f"R2 Std Dev:    {std_cv_r2:.4f}")
print(f"Last Fold R2:  {last_fold_r2:.4f}")

# Extract and Retrain (Final Fit is done automatically by GridSearchCV if refit=True)
best_rf = grid_search.best_estimator_

# Final Evaluation on 40% Test Set
final_preds = best_rf.predict(X_test)

print(f"\nTest MAPE: {mean_absolute_percentage_error(y_test, final_preds):.4f}")
print(f"Test RMSE: {root_mean_squared_error(y_test, final_preds):.4f}")
print(f"Test R2: {r2_score(y_test, final_preds):.4f}")

In [ ]:
# Calculate metrics for the plot title
final_rmse = root_mean_squared_error(y_test, final_preds)

# Create the Comparison Plot
plt.figure(figsize=(12, 6))

# Plot actual values
plt.plot(y_test.index, y_test.values, label='Measured Foothills Influent Alkalinity', 
         color='blue', marker='o', markersize=4, alpha=0.8)

# Plot predicted values
plt.plot(y_test.index, final_preds, label='Soft Sensor Predicted Alkalinity (Optimized RF)', 
         color='orange', linestyle='--', linewidth=2)
plt.axhline(y=60, color='r', linestyle='--', linewidth=2)

plt.title(f"Alkalinity Soft Sensor Performance (Test RMSE: {final_rmse:.3} mg/L)")
plt.xlabel("Date")
plt.ylabel("Alkalinity (mg/L)")
plt.legend()
plt.grid(True, alpha=0.3)

# Tighten layout to prevent clipping
plt.tight_layout()
plt.savefig(r"C:\Users\jslawson\OneDrive - Denver Water\SCO\Source_water_early_warning_systems\Data\Figures\AlkalinityPredictionComparison.png")
plt.show()

In [ ]:
# Extract importance and pair with feature names
importance = pd.Series(best_rf.feature_importances_, index=X.columns)

# Sort and plot
importance.sort_values().plot(kind='barh', color='skyblue')
plt.title("Random Forest Feature Importance (MDI)")
plt.tight_layout()
plt.savefig(r"C:\Users\jslawson\OneDrive - Denver Water\SCO\Source_water_early_warning_systems\Data\Figures\AlkalinityFeatureImportance.png")
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    best_rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=2, scoring = 'neg_root_mean_squared_error'
)

sorted_importances_idx = result.importances_mean.argsort()
importances = pd.DataFrame(
    result.importances[sorted_importances_idx].T,
    columns=X.columns[sorted_importances_idx],
)
ax = importances.plot.box(vert=False, whis=10)
ax.set_title("Permutation Importances (test set)")
ax.axvline(x=0, color="k", linestyle="--")
ax.set_xlabel("Decrease in RMSE score")
ax.figure.tight_layout()
plt.savefig(r"C:\Users\jslawson\OneDrive - Denver Water\SCO\Source_water_early_warning_systems\Data\Figures\AlkalinityPermutationImportance.png")
plt.show()

In [ ]:
# Try catboost before moving onto classification model
from catboost import CatBoostRegressor

# 55/45 chronological split
X_train_cv, X_test, y_train_cv, y_test = train_test_split(
    X,
    y,
    test_size=0.45,
    shuffle=False
)

# Create weights
weights = np.where(y_train_cv < 60.0, 1.5, 1.0)

# CatBoost baseline
cat_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    loss_function='RMSE',
    random_state=42,
    verbose=100
)

cat_model.fit(
    X_train_cv,
    y_train_cv,
    sample_weight=weights
)

# Predictions
final_preds = cat_model.predict(X_test)

print(f"\nTest MAPE: {mean_absolute_percentage_error(y_test, final_preds):.4f}")
print(f"Test RMSE: {root_mean_squared_error(y_test, final_preds):.4f}")
print(f"Test R2:   {r2_score(y_test, final_preds):.4f}")

In [ ]:
importance = pd.Series(
    cat_model.feature_importances_,
    index=X_train_cv.columns
).sort_values(ascending=False)

print(importance)


In [ ]:
# Calculate metrics for the plot title
final_rmse = root_mean_squared_error(y_test, final_preds)

# Create the Comparison Plot
plt.figure(figsize=(12, 6))

# Plot actual values
plt.plot(y_test.index, y_test.values, label='Measured Foothills Influent Alkalinity', 
         color='blue', marker='o', markersize=4, alpha=0.8)

# Plot predicted values
plt.plot(y_test.index, final_preds, label='Soft Sensor Predicted Alkalinity (Base CatBoost)', 
         color='orange', linestyle='--', linewidth=2)

plt.title(f"Alkalinity Soft Sensor Performance  (Test RMSE: {final_rmse:.2} mg/L)")
plt.xlabel("Date")
plt.ylabel("Alklinity (mg/L)")
plt.legend()
plt.grid(True, alpha=0.3)

# Tighten layout to prevent clipping
plt.tight_layout()
plt.savefig(r"C:\Users\jslawson\OneDrive - Denver Water\SCO\Source_water_early_warning_systems\Data\Figures\AlkalinityPredictionComparison_CatBoost.png")
plt.show()

In [ ]:
import shap

explainer = shap.TreeExplainer(cat_model)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test)

In [ ]:
# Try classification for above/below 60
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score

In [ ]:
# Add binary column for prediction
combined_df_alk['Alk_binary'] = np.where(combined_df_alk['Alk'] < 60, 1, 0) 

# Assign predictors and target columns
features = ['Specific_Cond_Mean', 'pH_Median', 'month_sin', 'month_cos', 'flow_7day_avg', 'turb_3day', 'turb_flow', 'Dissolved_Oxygen_Mean']
target = 'Alk_binary' 

# Finalize the dataframe
df_model = combined_df_alk[features + [target]].dropna()

# Set X and y
X = df_model[features]
y = df_model[target]

In [ ]:
# Checking the size of the different classes
print(df_model['Alk_binary'].value_counts(normalize=True))

In [ ]:
print("Starting Grid Search...")

# 55/45 Split
X_train_cv, X_test, y_train_cv, y_test = train_test_split(
    X, y, test_size=0.45, shuffle=False
)

# Setup the Classifier
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=tscv,
    scoring='recall',
    n_jobs=-1,
    verbose=1
)

# Fit
grid_search.fit(X_train_cv, y_train_cv)

In [ ]:
# Extract the best model
best_clf = grid_search.best_estimator_
print(f"\nBest Parameters: {grid_search.best_params_}")

# Predict on the 45% Test Set
y_pred = best_clf.predict(X_test)

# Print Individual Scores
print("\n--- Model Performance Metrics ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f} (When it predicts <60, how often is it right?)")
print(f"Recall:    {recall_score(y_test, y_pred):.4f} (How many of the actual <60 days did it catch?)")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")

# Detailed Report (includes scores for both 0 and 1)
print("\n--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred))

# Confusion Matrix for operational clarity
cm = confusion_matrix(y_test, y_pred)
print(f"\nOut of {len(X_test)} samples:")
print(f"True Negatives (Correct >60): {cm[0][0]}")
print(f"False Negatives (Missed <60 event): {cm[1][0]}")
print(f"False Positives (Mistakenly predicted <60): {cm[0][1]}")
print(f"True Positives (Correct <60): {cm[1][1]}")

In [ ]:
# Get the raw probabilities for both classes [Class 0, Class 1]
# We only care about the probability of being '1' (Low Alkalinity)
y_probs = best_clf.predict_proba(X_test)[:, 1]

In [ ]:
# Create a results dataframe for the test set
results_df = pd.DataFrame({
    'Actual': y_test,
    'Probability': y_probs,
    'Prediction': (y_probs >= 0.5).astype(int)
}, index=X_test.index).sort_index()

# Plotting
plt.figure(figsize=(15, 6))

# Plot the raw probability line
plt.plot(results_df.index, results_df['Probability'], color='blue', alpha=0.4, label='Predicted Probability')

# Add the 0.5 Threshold line
plt.axhline(y=0.5, color='red', linestyle='--', label='Threshold')

# Fill the areas where the prediction is 'Low'
plt.fill_between(results_df.index, 0, 1, where=(results_df['Probability'] >= 0.5), 
                 color='green', alpha=0.1, label='Predicted < 60 mg/L')

# Correct Lows (True Positives)
tp = results_df[(results_df['Actual'] == 1) & (results_df['Prediction'] == 1)]
plt.scatter(tp.index, [1.05]*len(tp), color='green', marker='v', s=20, label='Actual Low (Caught)')

# Missed Lows (False Negatives)
fn = results_df[(results_df['Actual'] == 1) & (results_df['Prediction'] == 0)]
plt.scatter(fn.index, [1.05]*len(fn), color='red', marker='x', s=20, label='Actual Low (Missed)')

# Missed Lows (False Positives)
fn = results_df[(results_df['Actual'] == 0) & (results_df['Prediction'] == 1)]
plt.scatter(fn.index, [1.05]*len(fn), color='red', marker='v', s=20, label='Actual high (Missed)')

plt.title("Alkalinity Classification: Predicted Probability vs Actual")
plt.ylabel("Probability of Alk < 60 mg/L")
plt.ylim(0, 1.1)
plt.xlim(dt.date(2026, 5, 1), dt.date(2026, 8, 20))
plt.legend(loc='lower left', ncol=2)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(r"C:\Users\jslawson\OneDrive - Denver Water\SCO\Source_water_early_warning_systems\Data\Figures\AlkalinityClassificationPerformance_limit.png")
plt.show()

In [ ]:
# Extract importance and pair with feature names
importance = pd.Series(best_clf.feature_importances_, index=X.columns)

# Sort and plot
importance.sort_values().plot(kind='barh', color='skyblue')
plt.title("Random Forest Feature Importance (MDI)")
plt.tight_layout()
plt.savefig(r"C:\Users\jslawson\OneDrive - Denver Water\SCO\Source_water_early_warning_systems\Data\Figures\AlkalinityClassificationFeatureImportance.png")
plt.show()

In [ ]:
result = permutation_importance(
    best_clf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=2, scoring = 'f1'
)

sorted_importances_idx = result.importances_mean.argsort()
importances = pd.DataFrame(
    result.importances[sorted_importances_idx].T,
    columns=X.columns[sorted_importances_idx],
)
ax = importances.plot.box(vert=False, whis=10)
ax.set_title("Permutation Importances (test set)")
ax.axvline(x=0, color="k", linestyle="--")
ax.set_xlabel("Decrease in F1 score")
ax.figure.tight_layout()
plt.savefig(r"C:\Users\jslawson\OneDrive - Denver Water\SCO\Source_water_early_warning_systems\Data\Figures\AlkalinityClassificationPermutationImportance.png")
plt.show()